In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_2016_2022_bn.csv")

In [3]:
df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_lt

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,FEM,LFG,EPFRG,GGAG,EC,DI,SSB,COC,CPI,EUPC
0,7.86,6.65,14.45,2.74,4.9,4.4,7.98,46.0,19.18,14.74,...,74.3,0.01,312.00,25.0,78.802445,7.47,92.931332,0.68,59.0,1.155739e+08
1,7.07,5.67,13.27,2.90,4.7,4.4,7.78,45.0,19.50,14.91,...,75.5,-0.01,294.85,-79.0,78.802445,7.41,92.984744,0.52,59.0,1.529936e+08
2,6.15,5.41,11.12,2.40,4.7,4.6,7.18,45.0,19.73,15.05,...,76.7,0.00,-161.65,86.0,78.802445,7.50,93.053500,0.47,59.0,2.870759e+08
3,6.26,5.49,11.85,6.23,4.8,4.9,6.44,45.0,19.86,15.11,...,77.4,0.01,350.20,128.0,78.802445,7.50,93.175686,0.67,60.0,1.883568e+08
4,8.49,7.69,19.56,7.94,4.7,4.4,5.69,45.0,19.93,15.12,...,75.8,0.01,150.30,5.0,78.802445,7.13,93.273709,0.78,60.0,5.145216e+08
5,7.11,6.61,14.30,3.74,4.7,4.4,5.31,45.0,19.95,15.11,...,76.7,0.00,-35.80,33.0,78.802445,7.18,93.083794,0.82,61.0,4.056306e+08
6,5.95,5.48,11.87,3.41,4.7,4.4,5.46,45.0,19.86,15.05,...,78.6,0.02,151.65,33.0,78.802445,7.31,93.083794,0.75,62.0,3.312880e+08


In [5]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['EPFRG_2'] = lt['EPFRG'].shift(2)

In [6]:
lt = lt.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('EPFRG_2', 'EUPC'),
])

In [9]:
sm.edges

OutEdgeView([('EPFRG_2', 'EUPC')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lt_2016_2022.html")

Graphs/fully_connected_lt_2016_2022.html


In [11]:
bn = BayesianNetwork(sm)

In [18]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
   UR  URFF  YUA  FDI  QOR  QOA  EIAP  PDPS  P65  P014  ...  LFG  EPFRG  GGAG  \
2   0     0    0    0    0    1     2     0    0     0  ...    0      0     2   
3   1     1    0    2    0    1     2     0    0     1  ...    1      2     2   
4   2     2    2    2    0    0     1     0    2     2  ...    1      1     0   
5   2     2    2    1    0    0     0     0    2     1  ...    0      0     0   
6   0     0    1    0    0    0     0     0    0     0  ...    2      2     0   

   EC  DI  SSB  COC  CPI  EUPC  EPFRG_2  
2   0   2    0    0    0     0        2  
3   0   2    2    0    0     0        1  
4   0   0    2    2    0     2        0  
5   0   0    0    2    2     2        2  
6   0   1    0    1    2     1        0  

[5 rows x 33 columns]


In [19]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.7139179706573486 seconds
Processing fold 1 using 7 cores takes 2.948422908782959 seconds
Processing fold 2 using 7 cores takes 2.961909055709839 seconds
Processing fold 3 using 7 cores takes 2.921475887298584 seconds
Processing fold 4 using 7 cores takes 2.974797010421753 seconds
Baseline AUC: 0.225


In [20]:
edges_to_add = [('LV', 'EPFRG_2'), ('LV', 'EUPC')]
edges_to_remove = [('EPFRG_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [21]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_2016_2022.html")

Graphs/node_added_lt_2016_2022.html


In [22]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EPFRG_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.0942509174346924 seconds
Processing fold 1 using 7 cores takes 3.143846273422241 seconds
Processing fold 2 using 7 cores takes 3.028783082962036 seconds
Processing fold 3 using 7 cores takes 3.0173633098602295 seconds
Processing fold 4 using 7 cores takes 2.9850339889526367 seconds
AUC from adding LV between 'EPFRG_2' and 'EUPC': 0.2
